In [57]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

In [58]:
movies_df = pd.read_csv("../data/raw/movies.csv")
ratings_df = pd.read_csv("../data/raw/ratings.csv")

In [59]:
ratings_df.info()
print("#" * 40)
movies_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105339 entries, 0 to 105338
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     105339 non-null  int64  
 1   movieId    105339 non-null  int64  
 2   rating     105339 non-null  float64
 3   timestamp  105339 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.2 MB
########################################
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10329 entries, 0 to 10328
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  10329 non-null  int64 
 1   title    10329 non-null  object
 2   genres   10329 non-null  object
dtypes: int64(1), object(2)
memory usage: 242.2+ KB


In [60]:
ratings_train, ratings_test = train_test_split(
    ratings_df, test_size=0.2, random_state=42
)

print(f"Train set size: {len(ratings_train)}")
print(f"Test set size: {len(ratings_test)}")

Train set size: 84271
Test set size: 21068


### Model 1: Linear Regression with Polynomial Features

In [61]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures


# ============================================
# PREPROCESSING FOR LINEAR REGRESSION
# ============================================
def preprocess_for_linear_regression(
    ratings_df, movies_df, fit_encoders=True, encoders=None
):
    """
    Preprocessing for Linear Regression:
    - User and movie average ratings
    - Rating counts for regularization-like effect
    - Polynomial features for non-linear relationships
    - Standard scaling
    """
    movies_df = movies_df[movies_df["genres"] != "(no genres listed)"]
    if encoders is None:
        encoders = {}

    df = ratings_df.copy()

    # Feature 1: User statistics
    user_stats = (
        df.groupby("userId").agg({"rating": ["mean", "std", "count"]}).fillna(0)
    )
    user_stats.columns = ["user_avg_rating", "user_std_rating", "user_rating_count"]
    df = df.merge(user_stats, on="userId", how="left")

    # Feature 2: Movie statistics
    movie_stats = (
        df.groupby("movieId").agg({"rating": ["mean", "std", "count"]}).fillna(0)
    )
    movie_stats.columns = ["movie_avg_rating", "movie_std_rating", "movie_rating_count"]
    df = df.merge(movie_stats, on="movieId", how="left")

    # Feature 3: Genre one-hot encoding
    df = df.merge(movies_df[["movieId", "genres"]], on="movieId", how="left")
    genres_dummies = df["genres"].str.get_dummies(sep="|")
    df = pd.concat([df, genres_dummies], axis=1)

    # Feature 4: Time-based feature (using timestamp)
    df["rating_timestamp"] = pd.to_datetime(df["timestamp"], unit="s")
    df["rating_hour"] = df["rating_timestamp"].dt.hour
    df["rating_dayofweek"] = df["rating_timestamp"].dt.dayofweek

    # Feature 5: Interaction features
    df["user_movie_avg_interaction"] = df["user_avg_rating"] * df["movie_avg_rating"]

    # Select features
    feature_columns = [
        col
        for col in df.columns
        if col
        not in [
            "userId",
            "movieId",
            "rating",
            "timestamp",
            "genres",
            "rating_timestamp",
        ]
    ]

    X = df[feature_columns]
    y = df["rating"]

    if fit_encoders:
        # Create polynomial features
        poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
        X_poly = poly.fit_transform(X)

        # Scale features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_poly)

        encoders["poly"] = poly
        encoders["scaler"] = scaler
        encoders["feature_columns"] = feature_columns
    else:
        poly = encoders["poly"]
        scaler = encoders["scaler"]
        X_poly = poly.transform(X)
        X_scaled = scaler.transform(X_poly)

    return X_scaled, y, encoders


# Preprocess for Linear Regression
X_train_lr, y_train_lr, lr_encoders = preprocess_for_linear_regression(
    ratings_train, movies_df, fit_encoders=True
)
X_test_lr, y_test_lr, _ = preprocess_for_linear_regression(
    ratings_test, movies_df, fit_encoders=False, encoders=lr_encoders
)

print(f"Linear Regression features shape: {X_train_lr.shape}")

# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_lr, y_train_lr)
y_pred_lr = lr_model.predict(X_test_lr)

# Evaluate
print("=" * 60)
print("Model 1: Linear Regression with Polynomial Features")
print("=" * 60)
mse_lr = mean_squared_error(y_test_lr, y_pred_lr)
mae_lr = mean_absolute_error(y_test_lr, y_pred_lr)
r2_lr = r2_score(y_test_lr, y_pred_lr)
print(f"MSE: {mse_lr:.4f}")
print(f"MAE: {mae_lr:.4f}")
print(f"RMSE: {np.sqrt(mse_lr):.4f}")
print(f"R²: {r2_lr:.4f}")

Linear Regression features shape: (84271, 406)
Model 1: Linear Regression with Polynomial Features
MSE: 0.5044
MAE: 0.5213
RMSE: 0.7102
R²: 0.5342


### Model 2: Random Forest with SVD-like Features

In [62]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import TruncatedSVD


# ============================================
# PREPROCESSING FOR RANDOM FOREST
# ============================================
def preprocess_for_random_forest(
    ratings_df, movies_df, fit_encoders=True, encoders=None
):
    """
    Preprocessing for Random Forest:
    - Creates user-movie matrix for collaborative filtering features
    - SVD decomposition for latent factors
    - Genre encoding
    - Statistical features (no scaling needed for RF)
    """
    movies_df = movies_df[movies_df["genres"] != "(no genres listed)"]
    if encoders is None:
        encoders = {}

    df = ratings_df.copy()

    # Feature 1: Basic statistics
    user_stats = df.groupby("userId")["rating"].agg(["mean", "count", "std"]).fillna(0)
    user_stats.columns = ["user_avg_rating", "user_rating_count", "user_rating_std"]
    df = df.merge(user_stats, on="userId", how="left")

    movie_stats = (
        df.groupby("movieId")["rating"].agg(["mean", "count", "std"]).fillna(0)
    )
    movie_stats.columns = ["movie_avg_rating", "movie_rating_count", "movie_rating_std"]
    df = df.merge(movie_stats, on="movieId", how="left")

    # Feature 2: SVD latent features (collaborative filtering)
    if fit_encoders:
        # Create user-movie matrix
        user_movie_matrix = df.pivot_table(
            values="rating", index="userId", columns="movieId", fill_value=0
        )

        # Apply SVD
        svd = TruncatedSVD(n_components=20, random_state=42)
        user_factors = svd.fit_transform(user_movie_matrix)
        movie_factors = svd.components_.T

        # Create mappings
        user_factor_map = dict(zip(user_movie_matrix.index, user_factors))
        movie_factor_map = dict(zip(user_movie_matrix.columns, movie_factors))

        encoders["svd"] = svd
        encoders["user_factor_map"] = user_factor_map
        encoders["movie_factor_map"] = movie_factor_map
        encoders["user_movie_matrix"] = user_movie_matrix
    else:
        user_factor_map = encoders["user_factor_map"]
        movie_factor_map = encoders["movie_factor_map"]

    # Add latent features
    user_factors_array = np.array(
        [user_factor_map.get(uid, np.zeros(20)) for uid in df["userId"]]
    )
    movie_factors_array = np.array(
        [movie_factor_map.get(mid, np.zeros(20)) for mid in df["movieId"]]
    )

    for i in range(20):
        df[f"user_factor_{i}"] = user_factors_array[:, i]
        df[f"movie_factor_{i}"] = movie_factors_array[:, i]

    # Feature 3: Genre features
    df = df.merge(movies_df[["movieId", "genres"]], on="movieId", how="left")
    genres_dummies = df["genres"].str.get_dummies(sep="|")
    df = pd.concat([df, genres_dummies], axis=1)

    # Feature 4: Genre count
    df["num_genres"] = df["genres"].str.count("|") + 1

    # Feature 5: Rating deviation features
    df["user_rating_deviation"] = df["rating"] - df["user_avg_rating"]
    df["movie_rating_deviation"] = df["rating"] - df["movie_avg_rating"]

    # Select features
    feature_columns = [
        col
        for col in df.columns
        if col not in ["userId", "movieId", "rating", "timestamp", "genres"]
    ]

    X = df[feature_columns].fillna(0)
    y = df["rating"]

    if fit_encoders:
        encoders["feature_columns"] = feature_columns

    return X, y, encoders


# Preprocess for Random Forest
X_train_rf, y_train_rf, rf_encoders = preprocess_for_random_forest(
    ratings_train, movies_df, fit_encoders=True
)
X_test_rf, y_test_rf, _ = preprocess_for_random_forest(
    ratings_test, movies_df, fit_encoders=False, encoders=rf_encoders
)

print(f"Random Forest features shape: {X_train_rf.shape}")

# Train Random Forest
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(X_train_rf, y_train_rf)
y_pred_rf = rf_model.predict(X_test_rf)

# Evaluate
print("=" * 60)
print("Model 2: Random Forest with SVD Features")
print("=" * 60)
mse_rf = mean_squared_error(y_test_rf, y_pred_rf)
mae_rf = mean_absolute_error(y_test_rf, y_pred_rf)
r2_rf = r2_score(y_test_rf, y_pred_rf)
print(f"MSE: {mse_rf:.4f}")
print(f"MAE: {mae_rf:.4f}")
print(f"RMSE: {np.sqrt(mse_rf):.4f}")
print(f"R²: {r2_rf:.4f}")

# Feature importance
feature_importance = pd.DataFrame(
    {"feature": X_train_rf.columns, "importance": rf_model.feature_importances_}
).sort_values("importance", ascending=False)
print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

Random Forest features shape: (84271, 68)
Model 2: Random Forest with SVD Features
MSE: 0.0353
MAE: 0.1238
RMSE: 0.1880
R²: 0.9674

Top 10 Most Important Features:
                   feature  importance
66   user_rating_deviation    0.381744
67  movie_rating_deviation    0.280947
3         movie_avg_rating    0.083371
0          user_avg_rating    0.073956
7           movie_factor_0    0.014706
5         movie_rating_std    0.014372
13          movie_factor_3    0.013753
2          user_rating_std    0.012834
15          movie_factor_4    0.012699
1        user_rating_count    0.006930


### Model 3: Gradient Boosting with Time-Aware Features

In [63]:
from sklearn.ensemble import GradientBoostingRegressor


# ============================================
# PREPROCESSING FOR GRADIENT BOOSTING
# ============================================
def preprocess_for_gradient_boosting(
    ratings_df, movies_df, fit_encoders=True, encoders=None
):
    """
    Preprocessing for Gradient Boosting:
    - Time-based features
    - User and movie temporal patterns
    - Genre and year features
    - Target encoding
    - Rolling statistics
    """
    movies_df = movies_df[movies_df["genres"] != "(no genres listed)"]
    if encoders is None:
        encoders = {}

    df = ratings_df.copy()

    # Feature 1: Time-based features
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")
    df["rating_month"] = df["timestamp"].dt.month
    df["rating_day"] = df["timestamp"].dt.day
    df["rating_hour"] = df["timestamp"].dt.hour
    df["rating_dayofweek"] = df["timestamp"].dt.dayofweek
    df["rating_quarter"] = df["timestamp"].dt.quarter
    df["rating_year"] = df["timestamp"].dt.year

    # Feature 2: Extract year from movie title (if available)
    movies_df_copy = movies_df.copy()
    movies_df_copy["movie_year"] = movies_df_copy["title"].str.extract(r"\((\d{4})\)")
    movies_df_copy["movie_year"] = pd.to_numeric(
        movies_df_copy["movie_year"], errors="coerce"
    )
    movies_df_copy["movie_age"] = (
        2024 - movies_df_copy["movie_year"]
    )  # Adjust year as needed

    df = df.merge(
        movies_df_copy[["movieId", "movie_year", "movie_age"]], on="movieId", how="left"
    )

    # Feature 3: User temporal patterns
    user_temporal = (
        df.groupby(["userId", "rating_dayofweek"])["rating"]
        .mean()
        .unstack(fill_value=0)
    )
    user_temporal.columns = [f"user_dow_{i}_avg" for i in range(7)]
    df = df.merge(user_temporal, on="userId", how="left")

    user_monthly = (
        df.groupby(["userId", "rating_month"])["rating"].mean().unstack(fill_value=0)
    )
    user_monthly.columns = [f"user_month_{i}_avg" for i in range(1, 13)]
    df = df.merge(user_monthly, on="userId", how="left")

    # Feature 4: Rolling statistics (user's recent ratings trend)
    df = df.sort_values(["userId", "timestamp"])
    df["user_rating_rolling_5"] = df.groupby("userId")["rating"].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )
    df["user_rating_rolling_10"] = df.groupby("userId")["rating"].transform(
        lambda x: x.rolling(10, min_periods=1).mean()
    )

    # Feature 5: Genre features (one-hot)
    df = df.merge(movies_df[["movieId", "genres"]], on="movieId", how="left")
    genres_dummies = df["genres"].str.get_dummies(sep="|")
    df = pd.concat([df, genres_dummies], axis=1)

    # Feature 6: User genre preferences
    for genre in genres_dummies.columns:
        df_temp = df.copy()
        df_temp[f"{genre}_rating"] = df_temp[genre] * df_temp["rating"]
        user_genre_avg = df_temp.groupby("userId")[f"{genre}_rating"].sum() / (
            df_temp.groupby("userId")[genre].sum() + 1e-6
        )
        df[f"user_{genre}_preference"] = df["userId"].map(user_genre_avg)

    # Feature 7: Movie popularity metrics
    movie_popularity = df.groupby("movieId").size().reset_index(name="movie_popularity")
    df = df.merge(movie_popularity, on="movieId", how="left")

    # Feature 8: User activity metrics
    user_activity = df.groupby("userId").size().reset_index(name="user_activity")
    df = df.merge(user_activity, on="userId", how="left")

    # Feature 9: Basic statistics
    df["user_avg_rating"] = df.groupby("userId")["rating"].transform("mean")
    df["movie_avg_rating"] = df.groupby("movieId")["rating"].transform("mean")
    df["user_rating_std"] = df.groupby("userId")["rating"].transform("std").fillna(0)
    df["movie_rating_std"] = df.groupby("movieId")["rating"].transform("std").fillna(0)

    # Select features
    feature_columns = [
        col
        for col in df.columns
        if col
        not in ["userId", "movieId", "rating", "timestamp", "genres", "movie_year"]
    ]

    X = df[feature_columns].fillna(0)
    y = df["rating"]

    if fit_encoders:
        encoders["feature_columns"] = feature_columns

    return X, y, encoders


# Preprocess for Gradient Boosting
X_train_gb, y_train_gb, gb_encoders = preprocess_for_gradient_boosting(
    ratings_train, movies_df, fit_encoders=True
)
X_test_gb, y_test_gb, _ = preprocess_for_gradient_boosting(
    ratings_test, movies_df, fit_encoders=False, encoders=gb_encoders
)

print(f"Gradient Boosting features shape: {X_train_gb.shape}")

# Train Gradient Boosting
gb_model = GradientBoostingRegressor(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=6,
    min_samples_split=10,
    min_samples_leaf=4,
    subsample=0.8,
    max_features=0.8,
    random_state=42,
)
gb_model.fit(X_train_gb, y_train_gb)
y_pred_gb = gb_model.predict(X_test_gb)

# Evaluate
print("=" * 60)
print("Model 3: Gradient Boosting with Time-Aware Features")
print("=" * 60)
mse_gb = mean_squared_error(y_test_gb, y_pred_gb)
mae_gb = mean_absolute_error(y_test_gb, y_pred_gb)
r2_gb = r2_score(y_test_gb, y_pred_gb)
print(f"MSE: {mse_gb:.4f}")
print(f"MAE: {mae_gb:.4f}")
print(f"RMSE: {np.sqrt(mse_gb):.4f}")
print(f"R²: {r2_gb:.4f}")

Gradient Boosting features shape: (84271, 72)
Model 3: Gradient Boosting with Time-Aware Features
MSE: 0.3798
MAE: 0.4452
RMSE: 0.6163
R²: 0.6493


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# ============================================
# PYTORCH DATASET CLASS
# ============================================
class RatingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(
            y.values if hasattr(y, "values") else y, dtype=torch.float32
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# ============================================
# MLP MODEL
# ============================================
class RatingMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=[256, 128, 64], dropout_rate=0.3):
        super(RatingMLP, self).__init__()

        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim

        # Output layer
        layers.append(nn.Linear(prev_dim, 1))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x).squeeze(-1)


# ============================================
# PREPROCESSING FUNCTION (similar to linear regression)
# ============================================
def preprocess_for_mlp(ratings_df, movies_df, fit_encoders=True, encoders=None):
    """
    Preprocessing for MLP:
    - Same feature engineering as linear regression
    - Standard scaling (without polynomial features)
    """
    movies_df = movies_df[movies_df["genres"] != "(no genres listed)"]
    if encoders is None:
        encoders = {}

    df = ratings_df.copy()

    # Feature 1: User statistics
    user_stats = (
        df.groupby("userId").agg({"rating": ["mean", "std", "count"]}).fillna(0)
    )
    user_stats.columns = ["user_avg_rating", "user_std_rating", "user_rating_count"]
    df = df.merge(user_stats, on="userId", how="left")

    # Feature 2: Movie statistics
    movie_stats = (
        df.groupby("movieId").agg({"rating": ["mean", "std", "count"]}).fillna(0)
    )
    movie_stats.columns = ["movie_avg_rating", "movie_std_rating", "movie_rating_count"]
    df = df.merge(movie_stats, on="movieId", how="left")

    # Feature 3: Genre one-hot encoding
    df = df.merge(movies_df[["movieId", "genres"]], on="movieId", how="left")
    genres_dummies = df["genres"].str.get_dummies(sep="|")
    df = pd.concat([df, genres_dummies], axis=1)

    # Feature 4: Time-based feature
    df["rating_timestamp"] = pd.to_datetime(df["timestamp"], unit="s")
    df["rating_hour"] = df["rating_timestamp"].dt.hour
    df["rating_dayofweek"] = df["rating_timestamp"].dt.dayofweek

    # Feature 5: Interaction features
    df["user_movie_avg_interaction"] = df["user_avg_rating"] * df["movie_avg_rating"]

    # Select features
    feature_columns = [
        col
        for col in df.columns
        if col
        not in [
            "userId",
            "movieId",
            "rating",
            "timestamp",
            "genres",
            "rating_timestamp",
        ]
    ]

    X = df[feature_columns]
    y = df["rating"]

    # Scale features
    if fit_encoders:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        encoders["scaler"] = scaler
        encoders["feature_columns"] = feature_columns
    else:
        scaler = encoders["scaler"]
        X_scaled = scaler.transform(X)

    return X_scaled, y, encoders


# ============================================
# TRAINING FUNCTION
# ============================================
def train_mlp_model(
    X_train,
    y_train,
    X_val,
    y_val,
    input_dim,
    hidden_dims=[256, 128, 64],
    batch_size=256,
    epochs=100,
    learning_rate=0.001,
    patience=10,
):

    # Create datasets and dataloaders
    train_dataset = RatingDataset(X_train, y_train)
    val_dataset = RatingDataset(X_val, y_val)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Initialize model
    model = RatingMLP(input_dim, hidden_dims=hidden_dims)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    # Training loop with early stopping
    best_val_loss = float("inf")
    patience_counter = 0

    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch_X.size(0)

        train_loss /= len(train_loader.dataset)

        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item() * batch_X.size(0)

        val_loss /= len(val_loader.dataset)

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            # Save best model
            best_model_state = model.state_dict()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

        scheduler.step(val_loss)

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch [{epoch + 1}/{epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}"
            )

    # Load best model
    model.load_state_dict(best_model_state)
    return model


# ============================================
# MAIN EXECUTION
# ============================================

# Load data (assuming you have the same data loading as before)
movies_df = pd.read_csv("../data/raw/movies.csv")
ratings_df = pd.read_csv("../data/raw/ratings.csv")

ratings_train, ratings_test = train_test_split(
    ratings_df, test_size=0.2, random_state=42
)

# Preprocess for MLP
X_train_mlp, y_train_mlp, mlp_encoders = preprocess_for_mlp(
    ratings_train, movies_df, fit_encoders=True
)
X_test_mlp, y_test_mlp, _ = preprocess_for_mlp(
    ratings_test, movies_df, fit_encoders=False, encoders=mlp_encoders
)

print(f"MLP features shape: {X_train_mlp.shape}")

# Split training data into train and validation
X_train_mlp, X_val_mlp, y_train_mlp, y_val_mlp = train_test_split(
    X_train_mlp, y_train_mlp, test_size=0.1, random_state=42
)

# Train MLP model
mlp_model = train_mlp_model(
    X_train_mlp,
    y_train_mlp,
    X_val_mlp,
    y_val_mlp,
    input_dim=X_train_mlp.shape[1],
    hidden_dims=[256, 128, 64],
    batch_size=256,
    epochs=100,
    learning_rate=0.001,
    patience=10,
)

# Evaluate on test set
mlp_model.eval()
test_dataset = RatingDataset(X_test_mlp, y_test_mlp)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

y_pred_mlp = []
with torch.no_grad():
    for batch_X, _ in test_loader:
        outputs = mlp_model(batch_X)
        y_pred_mlp.extend(outputs.numpy())

y_pred_mlp = np.array(y_pred_mlp)

# Evaluate
print("=" * 60)
print("Model 2: Multi-Layer Perceptron (MLP) with PyTorch")
print("=" * 60)

mse_mlp = mean_squared_error(y_test_mlp, y_pred_mlp)
mae_mlp = mean_absolute_error(y_test_mlp, y_pred_mlp)
r2_mlp = r2_score(y_test_mlp, y_pred_mlp)
print(f"MSE: {mse_mlp:.4f}")
print(f"MAE: {mae_mlp:.4f}")
print(f"RMSE: {np.sqrt(mse_mlp):.4f}")
print(f"R²: {r2_mlp:.4f}")

MLP features shape: (84271, 28)
Epoch [10/100], Train Loss: 0.7559, Val Loss: 0.6282
Epoch [20/100], Train Loss: 0.6830, Val Loss: 0.6234
Epoch [30/100], Train Loss: 0.6491, Val Loss: 0.6170
Early stopping at epoch 33
Model 2: Multi-Layer Perceptron (MLP) with PyTorch
MSE: 0.5112
MAE: 0.5257
RMSE: 0.7150
R²: 0.5279
